# Random Forest Notebook

This notebook trains and evaluates only the random forest churn model.

Workflow:
1. Add the project root to `sys.path`
2. Load the processed dataset
3. Prepare features and target
4. Split the data into train and test sets
5. Build the random forest pipeline
6. Train the model
7. Evaluate the model
8. Plot the ROC curve

In [ ]:
# Import Path so the notebook can locate the repository root.
from pathlib import Path
# Import sys so the notebook can modify Python's module search path.
import sys

# The notebook lives inside `notebooks/`, so the parent directory is the
# project root that contains the `src/` package.
PROJECT_ROOT = Path.cwd().resolve().parent
# Add the project root once so imports like `from src.models import ...`
# work inside this notebook.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

## Load the processed dataset

This cell reads the processed CSV that already has excluded columns removed.

In [ ]:
# Pandas loads the processed dataset into a DataFrame.
import pandas as pd
# Import the processed dataset path used by the project.
from src.config import PROCESSED_DATA_PATH

# Read the processed churn dataset.
df = pd.read_csv(PROCESSED_DATA_PATH)

# Display the first few rows for inspection.
df.head()

## Prepare features and target

This step separates the feature matrix `X` from the target vector `y` and builds the preprocessing transformer.

In [ ]:
# Import preprocessing helpers from the project code.
from src.preprocessing import prepare_features, build_preprocessor

# Split the DataFrame into feature columns and the churn target.
X, y = prepare_features(df)
# Build the preprocessing transformer used inside the sklearn pipeline.
preprocessor = build_preprocessor()

# Show the feature columns that will be passed into the model.
X.head()

## Train/test split

This creates a reproducible stratified train/test split so the churn ratio stays similar in both sets.

In [ ]:
# Import sklearn's helper for splitting data into training and testing sets.
from sklearn.model_selection import train_test_split

# Create the train/test split.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Build and train the random forest model

This cell constructs the pipeline and fits it on the training data.

In [ ]:
# Import the project helper that creates the random forest pipeline.
from src.models import build_random_forest_model

# Build and fit the random forest model.
rf_model = build_random_forest_model(preprocessor)
rf_model.fit(X_train, y_train)

## Evaluate the random forest model

This cell prints the classification report and ROC-AUC on the held-out test set.

In [ ]:
# Import the shared evaluation helper so metrics are computed consistently.
from src.evaluate import evaluate_model

# Evaluate the trained random forest model on the test set.
rf_results = evaluate_model(rf_model, X_test, y_test)

print("Random Forest Results:")
print(rf_results["classification_report"])
print("ROC-AUC:", rf_results["roc_auc"])

## Plot the ROC curve

This final cell uses the model's predicted churn probabilities to plot the ROC curve.

In [ ]:
# Import the ROC helper and matplotlib for plotting.
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

# Get predicted probabilities for the positive class.
y_prob = rf_model.predict_proba(X_test)[:, 1]
# Compute ROC curve coordinates.
fpr, tpr, _ = roc_curve(y_test, y_prob)

# Plot the random forest ROC curve.
plt.plot(fpr, tpr)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Random Forest ROC Curve")
plt.show()